# PADI Image — Demo: Compute SI P-value

Pipeline: Generate data → Train DeepSVDD → Detect anomaly → Compute SI p-value


In [7]:
import sys, os
sys.path.insert(0, os.path.abspath('../../..'))

import numpy as np
import torch
from padi.PADI_image.si_padi import compute_p_value_one_vs_mean
from padi.PADI_image.gen_data import prepare_synthetic_global_anomaly
from padi.PADI_image.model import DeepSVDDOneClass
from padi.util import extract_patches_tensor, resolve_device


In [8]:
# --- CONFIGURATION ---
MODEL_FAMILY = 'deepsvdd'
DEVICE = resolve_device()
PATCH_SZ = 30
PATCH_STR = 30
IMG_SIZE_FULL = (300, 300)
REPDIM = 16
CHANNELS = (8, 16)
DELTA = 20.0
ALPHA = 0.05

model_path = f'../model/{MODEL_FAMILY}_patch_synthetic_global.pth'
print(f'Model: {MODEL_FAMILY}, Device: {DEVICE}')


Model: deepsvdd, Device: cuda


## 1. Build or Load Model

In [9]:
ModelClass = DeepSVDDOneClass

if os.path.exists(model_path):
    print(f'Loading model from {model_path}...')
    model = ModelClass.load_model(model_path, device=DEVICE)
else:
    print(f'Training {MODEL_FAMILY} from scratch...')

    train_full_ds, _, _ = prepare_synthetic_global_anomaly(
        img_size=IMG_SIZE_FULL,
        n_train=500, n_ref=500, n_test_normal=50, n_test_anomaly=50,
        delta=DELTA)

    train_patches = extract_patches_tensor(train_full_ds.data, PATCH_SZ, PATCH_STR)
    flat_patches = train_patches.view(-1, 1, PATCH_SZ, PATCH_SZ)
    train_patch_ds = torch.utils.data.TensorDataset(
        flat_patches,
        torch.zeros(len(flat_patches), dtype=torch.long),
        torch.zeros(len(flat_patches), dtype=torch.long),
        torch.arange(len(flat_patches)))

    model = ModelClass(device=DEVICE)
    model.set_network(in_channels=1, img_size=(PATCH_SZ, PATCH_SZ),
                      repdim=REPDIM, channels=CHANNELS)
    model.pretrain(train_patch_ds, train_patch_ds,
                   n_epochs=100, batch_size=64)
    model.train(train_patch_ds, n_epochs=100, batch_size=64)

    # Compute R²
    model.net.eval()
    c_tensor = torch.tensor(model.c, device=DEVICE, dtype=torch.float32)
    all_dists = []
    with torch.no_grad():
        for x, _, _, _ in torch.utils.data.DataLoader(train_patch_ds, batch_size=64):
            out = model.net(x.to(DEVICE))
            dist = torch.sum((out - c_tensor) ** 2, dim=1)
            all_dists.append(dist.cpu().numpy())
    model.R_squared = float(np.quantile(np.concatenate(all_dists), 0.99))

    os.makedirs('../model', exist_ok=True)
    model.save_model(model_path)
    print(f'Model saved to {model_path}')

print(f'R² = {model.R_squared}')


Loading model from ../model/deepsvdd_patch_synthetic_global.pth...
R² = 2.2749371601094027e-05


## 2. Generate Test & Reference Data

In [10]:
_, test_full_ds, ref_full_ds = prepare_synthetic_global_anomaly(
    img_size=IMG_SIZE_FULL,
    n_train=10, n_ref=500, n_test_normal=50, n_test_anomaly=50,
    delta=DELTA)
print(f'Test: {len(test_full_ds)}, Ref: {len(ref_full_ds)}')


Test: 100, Ref: 500


## 3. Detect Anomaly & Compute P-value

In [11]:
img_shape_patch = (1, PATCH_SZ, PATCH_SZ)
d = PATCH_SZ * PATCH_SZ

# Random crop position
x = np.random.randint(IMG_SIZE_FULL[1] - PATCH_SZ + 1)
y = np.random.randint(IMG_SIZE_FULL[0] - PATCH_SZ + 1)

c_tensor = torch.tensor(model.c, device=DEVICE, dtype=torch.float32)

# Helper: compute SI p-value for a patch
def compute_si_for_patch(patch):
    N_REFS_SAMPLE = min(50, len(ref_full_ds))
    ref_indices = np.random.choice(len(ref_full_ds), N_REFS_SAMPLE, replace=False)
    refs = np.array([
        ref_full_ds[r][0][:, y:y+PATCH_SZ, x:x+PATCH_SZ].numpy().flatten()
        for r in ref_indices
    ])
    sigma_normalized = ref_full_ds.pixel_std_normalized
    sigma = sigma_normalized ** 2 * np.eye(d)
    return compute_p_value_one_vs_mean(
        patch.numpy().flatten(), refs, sigma, model,
        model.R_squared, img_shape_patch)

# --- H\u2080: Normal data (expect p-value > \u03b1 \u2192 do NOT reject) ---
print("=" * 60)
print("H\u2080: Testing on NORMAL data")
print("=" * 60)

for idx in range(50):  # first 50 are normal
    img, label, _, _ = test_full_ds[idx]
    patch = img[:, y:y+PATCH_SZ, x:x+PATCH_SZ]

    with torch.no_grad():
        p_tensor = patch.unsqueeze(0).to(DEVICE).float()
        output = model.net(p_tensor)
        score = torch.sum((output - c_tensor) ** 2).item()

    if score <= model.R_squared:
        continue

    print(f"Sample {idx} (label=0): score={score:.6f} > R\u00b2={model.R_squared:.6f}")
    print(f"  \u2192 Detected as anomaly!")

    p_value = compute_si_for_patch(patch)

    print(f"  SI P-value: {p_value}")
    if p_value is not None:
        print(f"  Reject H\u2080 at \u03b1={ALPHA}? {'YES' if p_value < ALPHA else 'NO'}")
    break
else:
    print("No anomalies detected in H\u2080 data \u2014 FPR is zero (good!)")

# --- H\u2081: Anomalous data (expect p-value < \u03b1 \u2192 reject) ---
print()
print("=" * 60)
print("H\u2081: Testing on ANOMALOUS data")
print("=" * 60)

for idx in range(50, len(test_full_ds)):  # last 50 are anomalous
    img, label, _, _ = test_full_ds[idx]
    patch = img[:, y:y+PATCH_SZ, x:x+PATCH_SZ]

    with torch.no_grad():
        p_tensor = patch.unsqueeze(0).to(DEVICE).float()
        output = model.net(p_tensor)
        score = torch.sum((output - c_tensor) ** 2).item()

    if score <= model.R_squared:
        continue

    print(f"Sample {idx} (label=1): score={score:.6f} > R\u00b2={model.R_squared:.6f}")
    print(f"  \u2192 Detected as anomaly!")

    p_value = compute_si_for_patch(patch)

    print(f"  SI P-value: {p_value}")
    if p_value is not None:
        print(f"  Reject H\u2080 at \u03b1={ALPHA}? {'YES' if p_value < ALPHA else 'NO'}")
    break
else:
    print("No true anomalies detected \u2014 try increasing DELTA")


H₀: Testing on NORMAL data
Sample 11 (label=0): score=0.000025 > R²=0.000023
  → Detected as anomaly!


/tmp/ipykernel_25022/403912763.py:8: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  c_tensor = torch.tensor(model.c, device=DEVICE, dtype=torch.float32)


  SI P-value: 0.6557820304627342
  Reject H₀ at α=0.05? NO

H₁: Testing on ANOMALOUS data
Sample 50 (label=1): score=0.000723 > R²=0.000023
  → Detected as anomaly!
  SI P-value: 0.0
  Reject H₀ at α=0.05? YES
